# Tool Sequence Evaluation (LangSmith `evaluate`)

LangSmith dataset `tool_sequence_dataset` 에 대해 place agent 를 실행하고, 실제 tool 호출 시퀀스가 기대값(`answer`) 과 일치하는지 자동 평가.

**시퀀스 문법**: `A|B` = 순차, `A+B` = 병렬 (한 `AIMessage` 안의 tool_calls)

In [1]:
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / ".env")

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "place-agent-tool-eval")

from langchain_core.messages import AIMessage
from langsmith.evaluation import evaluate

from agents.places.workflow import graph

DATASET_NAME = "tool_sequence_2_dataset"

In [2]:
def extract_tool_sequence(messages) -> str:
    steps = []
    for msg in messages:
        if not isinstance(msg, AIMessage):
            continue
        calls = getattr(msg, "tool_calls", None) or []
        if not calls:
            continue
        steps.append("+".join(sorted({tc["name"] for tc in calls})))
    return "|".join(steps)


def _normalize(seq: str) -> str:
    return "|".join("+".join(sorted(s.split("+"))) for s in seq.split("|") if s)


def target(inputs: dict) -> dict:
    question = inputs["question"]
    state = graph.invoke({
        "query": question,
        "messages": [{"role": "user", "content": question}],
    })
    return {"answer": extract_tool_sequence(state["messages"])}


def tool_sequence_match(outputs: dict, reference_outputs: dict) -> bool:
    return _normalize(reference_outputs.get("answer", "")) == _normalize(outputs.get("answer", ""))

In [3]:
results = evaluate(
    target,
    data=DATASET_NAME,
    evaluators=[tool_sequence_match],
    experiment_prefix="tool-seq",
    max_concurrency=1,
    
)
results

/Users/baeseong-yeon/Desktop/2026-1/bible-atlas-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'tool-seq-6f19b3b8' at:
https://smith.langchain.com/o/2da836b0-cc12-4655-aa17-2d2bc25c6cc9/datasets/8054f048-565c-456f-ad04-e86192e7d445/compare?selectedSessions=af716d95-bbf0-4db8-8850-2856217ca49b




7it [01:11, 13.67s/it]Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-SzWVzHR181kFB2oFO49ebYCD on tokens per min (TPM): Limit 30000, Used 28756, Requested 1721. Please try again in 954ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "/Users/baeseong-yeon/Desktop/2026-1/bible-atlas-agent/.venv/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1976, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/jq/281ljnr917zdxyy_68j2_0tr0000gn/T/ipykernel_23130/1812602286.py", line 19, in target
    state = graph.invoke({
            ^^^^^^^^^^^^^^
  File "/Users/baeseong-yeon/Desktop/2026-1/bible-atlas-agent/.venv/lib/python3.12/site-packages/langgraph/pregel/main.py", line 3913, in invoke
    for chunk in self.stream(
                 ^^^^^^^^^^^^

<ExperimentResults tool-seq-6f19b3b8>